[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/condensate/condensate.ipynb)

# A Rotating Bose–Einstein Condensate

This example computes the ground state of a rotating Bose–Einstein condensate in a harmonic trap: a lattice of quantized vortices. A condensate is one quantum state shared by all its atoms, a wave over the trap. It cannot rotate as a body does: its flow is set by how the wave turns from place to place, so it circulates only around vortices, holes in its density around which the wave turns once, each holding one quantum of circulation. Set rotating, the condensate holds as many vortices as the rotation asks, spread so evenly that on average it turns like a rigid body: twice the rotation rate over the circulation quantum of them per unit area, Feynman's rule, as [Donnelly (2009)](https://doi.org/10.1063/1.3248499) gives it for a rotating bucket of superfluid helium.

The algebra is the plane of the trap joined with a plane of the condensate's own, `Algebra("x+y+") * Algebra("a+b+")`. The condensate at each point of the trap is modelled as a vector in its own plane, whose length is the square root of the density and whose direction, its phase, says how far the wave has turned. Two turnings meet in the rotating trap, each a bivector of its own plane: the trap's, `mv.xy`, which drags each point a quarter turn about the middle, `point.commutator(mv.xy)`, and the wave's, `mv.ab`, which turns the condensate in its own plane, `wave * mv.ab`, and commutes with everything in the trap's plane. The energy's rotating part pairs them in plain sight: the wave's change along the trap's drag, read from its gradient in the mixed planes `xa xb ya yb`, against the wave turned a quarter in its own plane. The ground state is found by descending the energy at a fixed number of atoms, and the vortices are not put in anywhere: they emerge from the descent. A vortex is a triangle around which the wave turns once: the turns along its three edges, each the logarithm of the rotor between the wave's directions at the edge's ends, add up to a whole turn, with no angle unwrapped anywhere.

In the notation of the Gross–Pitaevskii equation the condensate is a complex function $\psi$, whose phase is the wave's direction in its plane and whose multiplication by $i$ is the quarter turn `wave * mv.ab`; with lengths in the trap's oscillator length and energies in its quantum, $\psi$ minimises $E = \int \tfrac12|\nabla\psi|^2 + \tfrac12 r^2|\psi|^2 + \tfrac g2|\psi|^4 - \Omega\,\psi^*L_z\psi$ at $\int|\psi|^2 = 1$, with $L_z = -i(x\partial_y - y\partial_x)$, and its vortices fill the cloud at $n_v = 2\Omega/\kappa = \Omega/\pi$ per unit area. There one $i$ stands for both turnings at once: the operator $x\partial_y - y\partial_x$ is the trap's turning of vertices, and the $i$ in front of it the wave's quarter turn, two rotations of two planes in one symbol.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext
from numga.sparse import spdiag
from examples.quantum.condensate import render, scenarios
from examples.util.animation import show_animation

# The trap's plane, and the condensate's own plane.
ga = Algebra("x+y+") * Algebra("a+b+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Plane = ga.gatype.vector()                                                    # Vector+
Point = ga.gatype.antivector()                                                # Antivector-
Wave = Plane.derive.within(Algebra("a+b+"))                                   # Vector+
Turn = ga.gatype.bivector().derive.within(Algebra("a+b+"))                    # Bivector+
# Unit steps across and up the trap's plane, as points from its middle.
axis = mv.x.dual()                                                            # [] Point-
across = mv.y.dual()                                                          # [] Point-
# The condensate's own plane, the join of its two directions, which no step across the trap has a part along.
wave_plane = mv.a.dual() & mv.b.dual()                                        # [] Antibivector-

## Settling into the lattice

The trap is a disk of triangles in rings, its vertices directions from the middle, and the condensate starts as a still cloud, a broad bell whose wave points nearly the same way everywhere, stirred by a little noise. Each step descends the condensate's energy: the kinetic energy, the trap and the rotation's drag in one implicit step, a map of waves factored once, and the atoms' repulsion explicitly; then the wave is rescaled to one atom. A hat varies with place in the trap and not in the condensate's own plane, so a corner's hat gradient is the plane through the facing side and the condensate's plane, over twice the triangle's area, `(wave_plane & sides) / double_areas`, and the kinetic energy reads each hat's gradient against the flow another's drives, `gradients & mobility(gradients)`: the mobility, the condensate's inverse mass, `Plane.dual()` in these units, is where the metric enters. The repulsion's strength and the rotation rate, nine tenths of the trap's frequency, come from the scenario. This cell takes about ten seconds, most of it drawing the animation.

In [ ]:
# The trap's disk, its vertices, and the trap's potential, half the squared distance from the middle.
topology, coordinates = scenarios.trap()
vertices = (axis * coordinates[:, 0] + across * coordinates[:, 1]).field()     # Point-[V]
potential = mv.scalar((0.5 * (coordinates ** 2).sum(axis=-1))[:, None]).field()   # Scalar+[V]
corners = topology.incidence[2, 0]                                            # [3] Scalar+[F <- V]
sides = topology.boundary[2, 1] * (topology.d[0] * vertices)[..., None]      # [3] Point-[F]
# Twice a triangle's area: two of its sides joined with the condensate's own plane.
double_areas = (sides[..., 0] & sides[..., 1]) & wave_plane                   # Scalar-[F]
areas = double_areas.norm() / 2                                               # Scalar+[F]
gradients = (wave_plane & sides) / double_areas[..., None]                    # [3] Plane+[F]
cells = (~corners * (areas / 3)).sum(axis=-1)                                 # Scalar+[V]

# The condensate's mobility, its inverse mass, one in these units: it turns a gradient of the wave into the flow it
# drives.
mobility = Plane.dual()                                                       # [] Point- <- Plane+


def settling(topology, vertices: Point, areas: Scalar, gradients: Plane, cells: Scalar, potential: Scalar,
             rotation: float, repulsion: float, wave: Wave, step: float, n_steps: int, stride: int) -> Iterator[Wave]:
    """The condensate `Wave+[V]` every `stride` steps of the descent, from this one, at one atom."""
    corners = topology.incidence[2, 0]                                        # [3] Scalar+[F <- V]

    def assembled(local):
        """A form on each triangle's corners gathered onto the vertices it joins."""
        return (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=-1).sum(axis=-1)

    # The kinetic energy: each hat's gradient read against the flow another's drives, over the triangle.
    kinetic = (gradients[..., :, None] & mobility(gradients[..., None, :])) * areas[..., None, None] / 2
    # The rotation drags each triangle's middle a quarter turn in the trap's plane; the wave's change along the drag is
    # the drag paired with each corner's hat gradient, times the corner's wave, shared out to the corners' cells.
    drags = (corners.mean(axis=0) * vertices).commutator(mv.xy)               # Point-[F]
    along = (areas / 3)[..., None, None] * (drags[..., None, None] & gradients[..., None, :])
    # The implicit step: the cells' wave over the step, the kinetic energy, the trap, and the rotation's drag turned a
    # quarter in the condensate's plane.
    stepping = (spdiag((cells / step + cells * potential) * Wave) + assembled(kinetic * Wave) +
                assembled(rotation * along * (Wave * mv.ab)))                 # (Wave+ <- Wave+)[V <- V]
    for index in range(n_steps + 1):
        if index % stride == 0:
            yield wave
        density = wave.scalar_norm_squared()                                  # Scalar+[V]
        wave = stepping.solve(cells * (wave / step - repulsion * density * wave))
        # Back to one atom.
        wave = wave / np.sqrt((cells * wave.scalar_norm_squared()).sites.sum().to_array())


# The still cloud: each vertex's amplitude times the plane's first direction turned by its angle.
amplitudes, angles = scenarios.still_cloud(coordinates)                       # [V], [V]
still = (amplitudes * ((mv.ab * (-angles / 2)).exp() >> mv.a)).field()        # Wave+[V]
frames = list(settling(topology, vertices, areas, gradients, cells, potential, scenarios.rotation, scenarios.repulsion,
                       still, scenarios.step, scenarios.n_steps, scenarios.stride))
# The frames over the vortices' coming in and settling.
drawn = frames[:scenarios.n_drawn_frames]
show_animation(render.animate(topology, vertices, (wave.scalar_norm_squared() for wave in drawn)), "condensate",
               scenarios.frame_duration_ms)

The condensate's density, deep blue where the atoms crowd and white where there are none. Vortices come in from the cloud's thin edge, white holes, and draw together into a lattice of near-equilateral triangles that turns with the trap. The animation shows the first half of the descent, where all of this happens and the lattice then barely moves; the vortices are counted below in the state at the descent's end.

## Feynman's rule

A vortex is a triangle around which the wave winds a whole turn, read from the logarithms of the rotors along its edges. Counted within each radius of the trap's middle, the vortices follow Feynman's rule, the rotation rate over pi of them per unit area, out to where the cloud thins.

In [ ]:
def windings(topology, wave: Wave) -> Turn:
    """Each triangle's winding `Turn+[F]`, in whole turns: the wave's turns along its edges, the logarithms of
    the rotors between the wave's directions at each edge's ends, added around it over a whole turn."""
    ends = topology.incidence[1, 0] * wave[..., None]                         # [2] Wave+[E]
    turns = (ends[..., 0] * ends[..., 1]).normalized().log()                  # Turn+[E]
    return (topology.boundary[2, 1] * turns[..., None]).sum(axis=-1) / (2 * np.pi)


settled = frames[-1]
middles = corners.mean(axis=0) * vertices                                     # Point-[F]
# Each triangle's winding in whole turns, read by the condensate's plane, and its middle's distance from the trap's.
turns = windings(topology, settled).scalar_product(~mv.ab)                    # Scalar+[F]
vortices = (turns > scenarios.vortex_share) | (turns < -scenarios.vortex_share)   # [F]
radii = scenarios.counting_radii                                              # [radii]
counted = ((middles.norm() <= radii[:, None]) & vortices).sum(axis=-1).reshape(len(radii))   # [radii]
render.draw_count(radii, counted, scenarios.rotation * radii ** 2);

The steps are the vortices counted within each radius, in oscillator lengths, and the curve is Feynman's rule, the rotation rate times the squared radius. The count follows the rule through the cloud's bulk and levels off at its edge, beyond which there are no atoms to circulate.

In [ ]:
# checks
# The condensate holds one atom.
np.testing.assert_allclose((cells * settled.scalar_norm_squared()).sites.sum().to_array(), 1.0, rtol=1e-10)
# Every vortex winds once, and all the same way, with the rotation.
whole = turns.to_array()[vortices]
np.testing.assert_allclose(np.abs(whole), 1.0, atol=1e-6)
assert len(np.unique(np.sign(whole))) == 1
# Within the cloud's bulk the vortices follow Feynman's rule within a fifth.
bulk = radii == 4.0
np.testing.assert_allclose(counted[bulk], scenarios.rotation * radii[bulk] ** 2, rtol=0.2)